# 14 · Rate limits and retries
Level L2 · Part 2 · core · ~35 min · builds on 13

## Goal
Write `call_with_retries`, which survives 429s, 5xx errors and timeouts with jittered exponential
backoff that honours `retry-after`, and refuses to retry a 400 or a spend-cap 429. Know what the
Anthropic and Gemini SDKs already do for you, and what they get wrong.

## The idea
Every provider caps how much you can send: a **rate limit**, counted in requests per minute (RPM) and
tokens per minute (TPM). Go over it and the server answers **429 Too Many Requests**, often with a
`retry-after` header saying how many seconds to wait. A 429 or a 5xx is temporary, so you wait and try
again. Wait a little longer each time (**exponential backoff**), and add randomness (**jitter**) so many
clients do not all come back at the same moment. Some errors never get better by waiting: a 400 (your
request is wrong, lesson 13) and a **spend cap** 429 (your monthly budget is used up).

Analogy: a busy restaurant. "Table in 10 minutes" means come back in 10. "We are closed for the month"
means do not stand at the door knocking.

## Picture
```mermaid
flowchart TD
    S["send request"] --> R{"result"}
    R -->|200| OK["done"]
    R -->|"400 / spend cap"| STOP["stop: never retry"]
    R -->|"429 / 5xx / timeout"| W{"retry-after header?"}
    W -->|yes| A["wait that long"]
    W -->|no| B["wait random 0..base x 2^attempt"]
    A --> S
    B --> S
```

## Step by step
### 1. A flaky provider on your laptop
We cannot make a real provider fail on demand, so Scout gets a **stub server**: a small local HTTP
server that answers like Anthropic or Gemini but fails on purpose. Each scenario is a path prefix:
- `/flaky/...`: 429 with `retry-after: 1` twice, then 200
- `/spendcap/...`: always the spend-cap 429 (body copied from Anthropic's docs), no `retry-after`
- `/badreq/...`: always 400
- `/slow/...`: the first call hangs for 2 seconds, then answers

It records the arrival time of every request at `/hits`. First, the folder for the file.

In [1]:
from pathlib import Path
Path("../apps/14").mkdir(parents=True, exist_ok=True)
print("ready to write ../apps/14/stub_server.py")

ready to write ../apps/14/stub_server.py


The server uses only the standard library. Read `do_POST`: that is where each scenario decides what to
send back. Paths containing `generateContent` get Gemini-shaped bodies, all others Anthropic-shaped.

In [2]:
%%writefile ../apps/14/stub_server.py
"""Scout's flaky-provider stub: answers like Anthropic or Gemini, but fails on purpose."""
import json, sys, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

PORT = int(sys.argv[1]) if len(sys.argv) > 1 else 8714
HITS: dict[str, list[float]] = {}          # scenario -> arrival times
START = time.monotonic()
TEXT = "Perseverance landed on Mars in February 2021."

def ok_body(gemini):
    if gemini:
        return {"candidates": [{"content": {"role": "model", "parts": [{"text": TEXT}]},
                                "finishReason": "STOP"}], "responseId": "stub-ok"}
    return {"id": "msg_stub", "type": "message", "role": "assistant", "model": "stub",
            "content": [{"type": "text", "text": TEXT}], "stop_reason": "end_turn",
            "stop_sequence": None, "usage": {"input_tokens": 20, "output_tokens": 12}}

def err_body(gemini, code, kind, msg, details=None):
    if gemini:
        return {"error": {"code": code, "message": msg, "status": kind}}
    err = {"type": kind, "message": msg} | ({"details": details} if details else {})
    return {"type": "error", "error": err, "request_id": "req_stub"}

class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):            # keep the notebook output clean
        pass

    def send(self, code, body, headers=None):
        data = json.dumps(body).encode()
        self.send_response(code)
        self.send_header("content-type", "application/json")
        for k, v in (headers or {}).items():
            self.send_header(k, v)
        self.send_header("content-length", str(len(data)))
        self.end_headers()
        try:
            self.wfile.write(data)
        except BrokenPipeError:              # the client timed out and left
            pass

    def do_GET(self):
        if self.path == "/health":
            return self.send(200, {"ok": True})
        if self.path == "/hits":
            return self.send(200, HITS)
        self.send(404, {"error": "not found"})

    def do_POST(self):
        self.rfile.read(int(self.headers.get("content-length", 0)))
        if self.path == "/reset":
            HITS.clear()
            return self.send(200, {"ok": True})
        scenario = self.path.strip("/").split("/")[0]
        n = len(HITS.setdefault(scenario, [])) + 1
        HITS[scenario].append(round(time.monotonic() - START, 3))
        g = "generateContent" in self.path
        if scenario == "flaky" and n <= 2:      # rate limited twice, then fine
            body = err_body(g, 429, "RESOURCE_EXHAUSTED" if g else "rate_limit_error",
                            "Rate limit exceeded: requests per minute.")
            return self.send(429, body, {"retry-after": "1"})
        if scenario == "spendcap":              # monthly cap reached: no retry-after
            body = err_body(g, 429, "rate_limit_error", "You have reached your API usage "
                            "limits. You will regain access on 2026-11-01 at 00:00 UTC.",
                            {"error_code": "enforced_spend_limit_reached"})
            return self.send(429, body)
        if scenario == "badreq":
            return self.send(400, err_body(g, 400, "invalid_request_error",
                                           "max_tokens: must be greater than 0"))
        if scenario == "slow" and n == 1:       # first call hangs for 2 s
            time.sleep(2)
        self.send(200, ok_body(g))

if __name__ == "__main__":
    ThreadingHTTPServer(("127.0.0.1", PORT), Handler).serve_forever()

Overwriting ../apps/14/stub_server.py


Start it on port 8714 (this lesson's fixed port), and poll `/health` until it answers instead of
sleeping a fixed time. `atexit` stops it even if a later cell fails.

In [3]:
import atexit, json, socket, subprocess, sys, time, urllib.request, urllib.error

PORT = 8714
BASE = f"http://127.0.0.1:{PORT}"
with socket.socket() as s:
    assert s.connect_ex(("127.0.0.1", PORT)) != 0, f"port {PORT} is busy"
server = subprocess.Popen([sys.executable, "../apps/14/stub_server.py", str(PORT)])
atexit.register(server.terminate)
for _ in range(50):
    try:
        print("health:", json.load(urllib.request.urlopen(BASE + "/health", timeout=1)))
        break
    except OSError:
        time.sleep(0.1)

health: {'ok': True}


Three small helpers: `post` sends JSON and returns `(status, headers, body)` for any status, `reset`
clears the server's log, and `gaps` reads it back as the seconds between attempts.

In [4]:
def post(path, payload=None, timeout=5.0):
    req = urllib.request.Request(BASE + path, data=json.dumps(payload or {}).encode(),
                                 headers={"content-type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, {k.lower(): v for k, v in r.headers.items()}, json.load(r)
    except urllib.error.HTTPError as e:   # 4xx/5xx still carry headers and a JSON body
        return e.code, {k.lower(): v for k, v in e.headers.items()}, json.load(e)

def reset():
    post("/reset")

def gaps(scenario):
    t = json.load(urllib.request.urlopen(BASE + "/hits"))[scenario]
    return [round(b - a, 1) for a, b in zip(t, t[1:])]

print(post("/badreq/v1/messages")[0])

400


### 2. How big is a rate limit?
Limits are per model and per organisation (Anthropic) or project (Gemini). Anthropic's Start tier gives
each model class 1,000 RPM and 2,000,000 input tokens per minute (ITPM). Which one Scout hits first
depends on how big each call is.

In [5]:
RPM, ITPM = 1_000, 2_000_000           # Anthropic Start tier, per model class
for tokens_per_call in (500, 3_000, 20_000):   # short chat, chat with notes, long agent run
    calls_by_tokens = ITPM // tokens_per_call
    limit = min(RPM, calls_by_tokens)
    binding = "RPM" if RPM <= calls_by_tokens else "TPM"
    print(f"{tokens_per_call:>6} tokens/call -> at most {limit:>5} calls/min ({binding} binds)")

   500 tokens/call -> at most  1000 calls/min (RPM binds)
  3000 tokens/call -> at most   666 calls/min (TPM binds)
 20000 tokens/call -> at most   100 calls/min (TPM binds)


Bigger prompts hit the token limit long before the request limit. Limits are also enforced over
shorter windows: Anthropic uses a **token bucket** and says 60 RPM may be enforced as 1 per second, so a
burst can get a 429 well below the per-minute number.

### 3. What a 429 looks like
One raw request to the flaky scenario. Look at the status, the `retry-after` header and the body.

In [6]:
reset()
status, headers, body = post("/flaky/v1/messages", {"prompt": "Summarise note 5."})
print("status     :", status)
print("retry-after:", headers.get("retry-after"))
print("body       :", body)

status     : 429
retry-after: 1
body       : {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': 'Rate limit exceeded: requests per minute.'}, 'request_id': 'req_stub'}


Now the spend-cap 429 next to it. Same status, same error type, but no `retry-after`, and a
`details.error_code` that says the money is gone until next month.

In [7]:
status, headers, body = post("/spendcap/v1/messages", {"prompt": "Summarise note 5."})
print("status     :", status, "| type:", body["error"]["type"])
print("retry-after:", headers.get("retry-after"))
print("error_code :", body["error"]["details"]["error_code"])

status     : 429 | type: rate_limit_error
retry-after: None
error_code : enforced_spend_limit_reached


Anthropic's docs are blunt: retrying this, including the SDK's automatic retries, fails until access
resumes. The status code alone cannot tell the two apart. Only `error_code` can.

### 4. Decide: retry or not?
One function makes the decision, so every caller agrees. Timeouts (no status at all), 408, 429 and
5xx are worth another try. The spend cap is checked **before** the generic 429 rule.

In [8]:
def retry_decision(status, body) -> tuple[bool, str]:
    if status == 200:
        return False, "ok"
    if status is None:
        return True, "timeout or network error"
    details = (body.get("error") or {}).get("details") or {}
    if isinstance(details, dict) and details.get("error_code") == "enforced_spend_limit_reached":
        return False, "spend cap reached: wait for next month"
    if status in (408, 429) or status >= 500:
        return True, f"{status}: temporary"
    return False, f"{status}: fix the request"

Try it on every case, including a 503 (server overloaded) built by hand.

In [9]:
spend_body = post("/spendcap/v1/messages")[2]
cases = [("200", 200, {}), ("429 rate limit", 429, {"error": {"type": "rate_limit_error"}}),
         ("429 spend cap", 429, spend_body), ("400 bad request", 400, post("/badreq/v1/messages")[2]),
         ("503 overloaded", 503, {}), ("timeout", None, {})]
for name, status, body in cases:
    print(f"{name:16} -> retry={retry_decision(status, body)}")

200              -> retry=(False, 'ok')
429 rate limit   -> retry=(True, '429: temporary')
429 spend cap    -> retry=(False, 'spend cap reached: wait for next month')
400 bad request  -> retry=(False, '400: fix the request')
503 overloaded   -> retry=(True, '503: temporary')
timeout          -> retry=(True, 'timeout or network error')


### 5. Exponential backoff with jitter
Without `retry-after`, wait longer after each failure: `base x 2^attempt` seconds, capped. Then pick a
random time between 0 and that number. This is **full jitter**. The random generator is seeded so the
numbers below are the same every run.

In [10]:
import random

def backoff_delay(attempt, base=0.5, cap=8.0, rng=random):
    return rng.uniform(0, min(cap, base * 2 ** attempt))

rng = random.Random(14)
for attempt in range(6):
    ceiling = min(8.0, 0.5 * 2 ** attempt)
    print(f"attempt {attempt}: ceiling {ceiling:4.1f}s -> wait {backoff_delay(attempt, rng=rng):.2f}s")

attempt 0: ceiling  0.5s -> wait 0.05s
attempt 1: ceiling  1.0s -> wait 0.70s
attempt 2: ceiling  2.0s -> wait 1.30s
attempt 3: ceiling  4.0s -> wait 3.76s
attempt 4: ceiling  8.0s -> wait 2.17s
attempt 5: ceiling  8.0s -> wait 2.05s


Why the randomness? Picture five Scout workers that all got a 429 at the same moment. With plain
backoff they all come back together and hit the limit together, again. With jitter they spread out.

In [11]:
rng = random.Random(14)
plain = [0.5 * 2 ** 1 for _ in range(5)]                       # second retry, no jitter
jittered = sorted(round(backoff_delay(1, rng=rng), 2) for _ in range(5))
print("no jitter  :", plain)
print("full jitter:", jittered)

no jitter  : [1.0, 1.0, 1.0, 1.0, 1.0]
full jitter: [0.11, 0.27, 0.65, 0.7, 0.94]


Five requests in the same instant is a burst: exactly what a token bucket rejects. This pile-up is
called a **thundering herd**.

### 6. `call_with_retries`
Now put it together. `send` is any function that returns `(status, headers, body)`, so the same loop
works for our raw HTTP and, later, for an SDK. It prefers the server's `retry-after` over its own
backoff, and stops after `max_attempts`.

In [12]:
class GiveUp(Exception):
    pass

def call_with_retries(send, max_attempts=4, rng=random.Random(14)):
    for attempt in range(max_attempts):
        try:
            status, headers, body = send()
        except (TimeoutError, urllib.error.URLError) as e:
            status, headers, body = None, {}, {"error": {"type": type(e).__name__}}
        retry, why = retry_decision(status, body)
        print(f"  attempt {attempt + 1}: {status} -> {why}")
        if status == 200:
            return body
        if not retry or attempt == max_attempts - 1:
            raise GiveUp(why)
        ra = headers.get("retry-after")
        wait = float(ra) if ra else backoff_delay(attempt, rng=rng)
        print(f"  waiting {wait:.2f}s ({'retry-after' if ra else 'backoff'})")
        time.sleep(wait)

Run it against the flaky scenario. It should fail twice, wait one second each time (the server said so),
then succeed.

In [13]:
reset()
body = call_with_retries(lambda: post("/flaky/v1/messages", {"prompt": "Summarise note 5."}))
print("answer:", body["content"][0]["text"])
print("gaps between attempts:", gaps("flaky"))

  attempt 1: 429 -> 429: temporary
  waiting 1.00s (retry-after)


  attempt 2: 429 -> 429: temporary
  waiting 1.00s (retry-after)


  attempt 3: 200 -> ok
answer: Perseverance landed on Mars in February 2021.
gaps between attempts: [1.0, 1.0]


And against the two errors that must not be retried. Each should reach the server exactly once.

In [14]:
reset()
for scenario in ("spendcap", "badreq"):
    try:
        call_with_retries(lambda: post(f"/{scenario}/v1/messages"))
    except GiveUp as e:
        print(f"{scenario}: gave up ({e})")
print("hits:", {k: len(v) for k, v in json.load(urllib.request.urlopen(BASE + "/hits")).items()})

  attempt 1: 429 -> spend cap reached: wait for next month
spendcap: gave up (spend cap reached: wait for next month)
  attempt 1: 400 -> 400: fix the request
badreq: gave up (400: fix the request)
hits: {'spendcap': 1, 'badreq': 1}


### 7. Timeouts
A provider can also just hang. Without a **timeout** your call waits forever. Here the first call to
`/slow` hangs for 2 seconds, our timeout is 0.5 seconds, so attempt 1 fails with `TimeoutError` and is
retried after a short backoff.

In [15]:
reset()
body = call_with_retries(lambda: post("/slow/v1/messages", timeout=0.5))
print("answer:", body["content"][0]["text"])

  attempt 1: None -> timeout or network error
  waiting 0.05s (backoff)
  attempt 2: 200 -> ok
answer: Perseverance landed on Mars in February 2021.


A timed-out request may still have run on the server, and you may still be billed for it. For a chat
reply that is fine; for a tool that sends an email it is not (lesson 39).

### 8. The Anthropic SDK retries for you
SDKs already retry. Anthropic's client retries connection errors, 408, 409, 429 and 5xx twice by
default (`max_retries=2`), with its own backoff, and it obeys `retry-after`. Point it at the stub with
`base_url` (the key `"stub"` is a dummy: nothing leaves your laptop).

In [16]:
import anthropic

claude = anthropic.Anthropic(api_key="stub", base_url=BASE + "/flaky")
reset()
msg = claude.messages.create(model="claude-haiku-4-5", max_tokens=60,
                             messages=[{"role": "user", "content": "Summarise note 5."}])
print("answer:", msg.content[0].text)
print("gaps between attempts:", gaps("flaky"))

answer: Perseverance landed on Mars in February 2021.
gaps between attempts: [1.0, 1.0]


Three hits, one second apart: the SDK read `retry-after`. Now the spend cap.

In [17]:
claude_cap = anthropic.Anthropic(api_key="stub", base_url=BASE + "/spendcap")
reset()
try:
    claude_cap.messages.create(model="claude-haiku-4-5", max_tokens=60,
                               messages=[{"role": "user", "content": "Summarise note 5."}])
except anthropic.RateLimitError as e:
    print(type(e).__name__, e.status_code, e.body["error"]["details"])
print("hits on spendcap:", len(json.load(urllib.request.urlopen(BASE + "/hits"))["spendcap"]))

RateLimitError 429 {'error_code': 'enforced_spend_limit_reached'}
hits on spendcap: 3


Three hits for an error that cannot get better. The SDK treats every 429 the same. The fix: switch its
retries off with `max_retries=0` and drive it with `call_with_retries`. A small adapter turns the SDK's
result or exception into `(status, headers, body)`.

In [18]:
def claude_send(client):
    def send():
        try:
            m = client.messages.create(model="claude-haiku-4-5", max_tokens=60,
                                       messages=[{"role": "user", "content": "Summarise note 5."}])
            return 200, {}, {"text": m.content[0].text}
        except anthropic.APIStatusError as e:
            return e.status_code, dict(e.response.headers), e.body or {}
    return send

Same two scenarios, now through our loop.

In [19]:
reset()
for path in ("/flaky", "/spendcap"):
    client = anthropic.Anthropic(api_key="stub", base_url=BASE + path, max_retries=0)
    try:
        print(path, "->", call_with_retries(claude_send(client)))
    except GiveUp as e:
        print(path, "-> gave up:", e)
print({k: len(v) for k, v in json.load(urllib.request.urlopen(BASE + "/hits")).items()})

  attempt 1: 429 -> 429: temporary
  waiting 1.00s (retry-after)


  attempt 2: 429 -> 429: temporary
  waiting 1.00s (retry-after)


  attempt 3: 200 -> ok
/flaky -> {'text': 'Perseverance landed on Mars in February 2021.'}
  attempt 1: 429 -> spend cap reached: wait for next month
/spendcap -> gave up: spend cap reached: wait for next month
{'flaky': 3, 'spendcap': 1}


### 9. Scout's Gemini client does not retry by default
Scout's own client (lesson 13) is `google-genai`. Its retries are **off** unless you pass
`HttpRetryOptions`. Here is the lesson 13 client pointed at the stub with `base_url`.

In [20]:
from google import genai
from google.genai import types, errors

def cfg(**kwargs) -> types.GenerateContentConfig:
    off = types.AutomaticFunctionCallingConfig(disable=True)
    return types.GenerateContentConfig(automatic_function_calling=off, **kwargs)

gem = genai.Client(vertexai=True, api_key="stub",
                   http_options=types.HttpOptions(base_url=BASE + "/flaky"))
reset()
try:
    gem.models.generate_content(model="gemini-3.5-flash-lite", contents="Summarise note 5.", config=cfg())
except errors.APIError as e:
    print(type(e).__name__, e.code, e.status, "| hits:", len(json.load(urllib.request.urlopen(BASE + "/hits"))["flaky"]))

ClientError 429 RESOURCE_EXHAUSTED | hits: 1


One hit, then a `ClientError` 429. Now switch retries on: up to 4 attempts, first wait 0.2 s, doubling,
with jitter. Its default list of retried codes is 408, 429, 500, 502, 503 and 504.

In [21]:
retry = types.HttpRetryOptions(attempts=4, initial_delay=0.2, max_delay=8.0, exp_base=2, jitter=0.2)
gem = genai.Client(vertexai=True, api_key="stub",
                   http_options=types.HttpOptions(base_url=BASE + "/flaky", retry_options=retry))
reset()
resp = gem.models.generate_content(model="gemini-3.5-flash-lite", contents="Summarise note 5.", config=cfg())
print("answer:", resp.text)
print("hits:", len(json.load(urllib.request.urlopen(BASE + "/hits"))["flaky"]),
      "| every gap shorter than retry-after (1 s):", all(g < 1.0 for g in gaps("flaky")))

answer: Perseverance landed on Mars in February 2021.
hits: 3 | every gap shorter than retry-after (1 s): True


It worked, but it ignored `retry-after: 1` and came back sooner, so against a real provider those early
retries could be rejected too. It also retries every 429, spend cap included. If that matters, set
`attempts=1` and use your own loop, as with Anthropic.

Stop the stub server.

In [22]:
server.terminate()
server.wait(timeout=5)
print("server stopped, exit code", server.returncode)

server stopped, exit code -15


## At work
- **Teams set retries explicitly and log each one.** SDK defaults differ (Anthropic: 2 retries,
  honours `retry-after`; `google-genai`: none), so production code pins `max_retries` or retry options
  in config and counts retries per call. → lesson 49
- **The spend cap is a real outage.** When an Anthropic organisation hits its monthly cap, every call
  returns 429 until 00:00 UTC on the first of next month; teams alert on `enforced_spend_limit_reached`
  and set their own lower limit, which returns a 400 instead. → lesson 54
- **Bursts break limits, not averages.** Limits use a token bucket and sudden ramps trigger
  "acceleration limits", so services that run many conversations cap their own concurrency. → lesson 38
- **Retries can duplicate side effects.** A timed-out call may have run; retrying a tool that writes or
  sends needs an idempotency key. → lesson 39

## What just happened
- A local stub server played a provider that rate-limits, hangs, rejects bad requests and hits a
  spend cap.
- The rate-limit 429 had `retry-after: 1`; the spend-cap 429 had no header and an `error_code` instead.
- `retry_decision` retried 429, 5xx and timeouts, and refused 400 and the spend cap.
- Seeded full jitter spread five simultaneous retries apart instead of sending them together.
- `call_with_retries` waited 1 s twice on `flaky`, then succeeded; `spendcap` and `badreq` hit once.
- Anthropic's SDK honoured `retry-after` but sent the spend-cap request 3 times; `max_retries=0` plus
  our loop fixed it.
- `google-genai` did not retry at all until given `HttpRetryOptions`, and then ignored `retry-after`.

## Common mistakes
The stub is stopped, so these use responses saved above. **1. Assuming every 429 has `retry-after`.**
The spend-cap 429 does not, so reading the header blindly crashes.

In [23]:
cap_headers = {"content-type": "application/json"}   # the spend-cap 429 headers from step 3
rl_headers = {"content-type": "application/json", "retry-after": "1"}
print("spend cap:", cap_headers, "| rate limit:", rl_headers)

spend cap: {'content-type': 'application/json'} | rate limit: {'content-type': 'application/json', 'retry-after': '1'}


The crash:

In [24]:
wait = float(cap_headers["retry-after"])

KeyError: 'retry-after'

Fix: check the decision first (the spend cap stops here), and use `.get` with a backoff fallback.

In [25]:
retry, why = retry_decision(429, spend_body)
print("retry?", retry, "|", why)
ra = rl_headers.get("retry-after")
print("rate-limit wait:", float(ra) if ra else backoff_delay(0))

retry? False | spend cap reached: wait for next month
rate-limit wait: 1.0


**2. Retrying every error.** A catch-all loop retries the 400 and then fails anyway, three calls later.
A fake `send` stands in for the stopped server.

In [26]:
calls = []
def send_bad():
    calls.append(1)
    raise ValueError("400 invalid_request_error: max_tokens: must be greater than 0")

for attempt in range(3):
    try:
        send_bad()
        break
    except Exception:
        continue
print("calls made:", len(calls))
send_bad()

calls made: 3


ValueError: 400 invalid_request_error: max_tokens: must be greater than 0

Fix: decide from the status before retrying, as `call_with_retries` does. A 400 is sent once.

In [27]:
print(retry_decision(400, {"error": {"type": "invalid_request_error"}}))

(False, '400: fix the request')


## Try it
A user will not wait a minute for a summary. Write `call_with_budget(send, max_wait_s)`: like
`call_with_retries`, but it raises `GiveUp("wait budget exhausted")` instead of sleeping when the next
wait would push the **total** waiting past `max_wait_s`.

Test it with the fake `send` below (no server needed): it returns 429 with `retry-after: 1` twice, then
200. With `max_wait_s=3` it should succeed; with `max_wait_s=1.5` it should give up after one wait.

In [28]:
def make_flaky():
    n = [0]
    def send():
        n[0] += 1
        if n[0] <= 2:
            return 429, {"retry-after": "1"}, {"error": {"type": "rate_limit_error"}}
        return 200, {}, {"text": "Perseverance landed on Mars in February 2021."}
    return send

# your code here

## Key terms
- **rate limit** — the provider's cap on how much you may send per time window; going over returns 429.
- **RPM / TPM** — requests per minute and tokens per minute; Anthropic splits TPM into input (ITPM) and output (OTPM).
- **token bucket** — a limiter that refills continuously; bursts can be rejected below the per-minute number.
- **429 Too Many Requests** — the HTTP status for "slow down"; temporary unless it is a spend cap.
- **retry-after** — a response header with the seconds to wait before retrying.
- **exponential backoff** — waiting `base x 2^attempt` seconds after each failure, up to a cap.
- **jitter** — randomness added to the wait; full jitter picks uniformly between 0 and the backoff.
- **thundering herd** — many clients retrying at the same moment and overloading the server again.
- **spend cap** — a monthly spending limit; Anthropic returns 429 `enforced_spend_limit_reached`, never worth retrying.
- **timeout** — the longest you wait for a response before treating the call as failed.